# ASIC Pipeline: Final Harmonized Data

This notebook runs the current ASIC harmonization pipeline from the codebase and displays the final static and dynamic outputs.

It intentionally imports the shared pipeline implementation instead of re-implementing any harmonization logic here.


In [1]:
from pathlib import Path
import sys

import pandas as pd

try:
    from IPython.display import Markdown, display
except ModuleNotFoundError:
    def display(obj):
        print(obj)

    def Markdown(text):
        return text

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 240)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "icu_data_platform").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing src/icu_data_platform")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from icu_data_platform.sources.asic.extract.raw_tables import (  # noqa: E402
    DEFAULT_ASIC_RAW_DATA_DIR,
    DEFAULT_TRANSLATION_PATH,
)
from icu_data_platform.sources.asic.pipeline import build_asic_harmonized_dataset  # noqa: E402

PROJECT_ROOT


PosixPath('/Users/joanameyer/repository/icu-data-platform')

In [2]:
PIPELINE_PARAMETERS = {
    "raw_dir": DEFAULT_ASIC_RAW_DATA_DIR,
    "translation_path": DEFAULT_TRANSLATION_PATH,
    "min_non_null": 20,
    "min_hospitals": 4,
    "fence_factor": 1.5,
}

if not PIPELINE_PARAMETERS["raw_dir"].exists():
    raise FileNotFoundError(f"ASIC raw data directory not found: {PIPELINE_PARAMETERS['raw_dir']}")

display(Markdown("## Pipeline Parameters"))
display(pd.Series({key: str(value) for key, value in PIPELINE_PARAMETERS.items()}, name="value").to_frame())

dataset = build_asic_harmonized_dataset(**PIPELINE_PARAMETERS)
final_static = dataset.static.combined.copy()
final_dynamic = dataset.dynamic.combined.copy()

display(Markdown("## Final Dataset Sizes"))
display(
    pd.DataFrame(
        [
            {
                "table": "static",
                "rows": final_static.shape[0],
                "columns": final_static.shape[1],
                "hospitals": final_static["hospital_id"].nunique(dropna=True),
            },
            {
                "table": "dynamic",
                "rows": final_dynamic.shape[0],
                "columns": final_dynamic.shape[1],
                "hospitals": final_dynamic["hospital_id"].nunique(dropna=True),
            },
        ]
    )
)


## Pipeline Parameters

,value
raw_dir,/Users/joanameyer/data/asic/raw_sample10
translation_path,/Users/joanameyer/repository/icu-data-platform/src/icu_data_platform/sources/asic/column_translation.json
min_non_null,20
min_hospitals,4
fence_factor,1.5


## Final Dataset Sizes

,table,rows,columns,hospitals
0,static,70,16,7
1,dynamic,101647,107,7


## Final Static Data

`final_static` contains the harmonized static table returned by the current pipeline.


In [3]:
display(Markdown("### Preview"))
display(final_static.head(10))

display(Markdown("### Rows Per Hospital"))
display(
    final_static.groupby("hospital_id", dropna=False)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values("hospital_id")
    .reset_index(drop=True)
)

display(Markdown("### Source Map Sample"))
display(dataset.static.source_map.head(20))

display(Markdown("### Schema Summary"))
display(dataset.static.schema_summary)


### Preview

,hospital_id,stay_id_global,stay_id_local,age_group,sex,height_group,weight_group,bmi_group,hosp_mortality,icu_mortality,hosp_los,icu_los,icu_readmit,dialysis_free_days,vent_free_days,icd10_codes
0,asic_UK00,asic_UK00_372,372,<70,F,<180,<65,Normal Weight,1,<NA>,<NA>,4,<NA>,<NA>,<NA>,"C34,C77,C78,C79,D25,D37,D61,D69,D70,E46,E83,E87,G83,I26,I74,J40,J90,J96,K08,L89,M16,M49,M89,R00,R04,R11,R40,R52,R63,U99,Z03,Z11,Z51,Z74,Z80,Z91,Z92"
1,asic_UK00,asic_UK00_419,419,<70,M,>185,76-250,Overweight,0,<NA>,<NA>,14,<NA>,<NA>,<NA>,"A09,A41,B95,B96,D62,D63,D68,E11,E87,F05,G61,G63,I10,I48,I79,J18,J91,J95,J98,K50,L03,L89,M62,M86,N08,N13,N17,N18,N39,R18,R33,R57,R63,R65,T25,T79,T87,U69,U81,U99,Z11,Z29"
2,asic_UK00,asic_UK00_1014,1014,<70,M,<180,76-250,Overweight,0,<NA>,<NA>,28,<NA>,<NA>,<NA>,"B97,D62,D65,E03,E87,E88,F05,I27,J12,J80.02,J96,J98,L89,N17,R13,R65,U07.1,U10,Z29,Z43"
3,asic_UK00,asic_UK00_1204,1204,<70,M,180-185,76-250,Overweight,0,<NA>,<NA>,4,<NA>,<NA>,<NA>,"B95,B96,D69,E11,E78,E79,G63,I21,I25,I72,J90,J96,K21,L97,L98,M14,M96,T84,U69,U99,Z11,Z92,Z98"
4,asic_UK00,asic_UK00_1955,1955,<70,M,180-185,76-250,Overweight,0,<NA>,<NA>,14,<NA>,<NA>,<NA>,"A41,D62,D68,F05,J15,J95,J98,R65,S01,S02,S21,S22,S25,S26,S27,S31,S32,S36,S37,S39,S42,S51,S52,S91,S93,T79,U69,U99,V99,Z11,Z93"
5,asic_UK00,asic_UK00_2203,2203,70-79,M,<180,76-250,Overweight,0,<NA>,<NA>,13,<NA>,<NA>,<NA>,"D68,E53,E87,F17,G81,G91,G94,H57,I10,I60,I61,I70,J15,J90,J93,J95,J96,J98,N18,U50,U52,U69,Z43,Z92,Z95"
6,asic_UK00,asic_UK00_2353,2353,<70,M,<180,76-250,Overweight,0,<NA>,<NA>,4,<NA>,<NA>,<NA>,"D62,D69,E78,I21,I25,I49,I50,J18,K08,U69,U99,Z03,Z11,Z92"
7,asic_UK00,asic_UK00_2578,2578,<70,M,<180,76-250,Overweight,1,<NA>,<NA>,15,<NA>,<NA>,<NA>,"A41,B95,B96,C34,C77,D62,D90,E11,E16,E78,E87,I07,I10,I25,I26,I27,I31,I34,J15,J80.03,J86,J91,J98,N17,R09,R57,R65,U69,Z22,Z43,Z92,Z93"
8,asic_UK00,asic_UK00_2989,2989,<70,F,<180,76-250,Obesity Class 3,1,<NA>,<NA>,3,<NA>,<NA>,<NA>,"A41,E87,J44,J80.03,N17,R57,R65,Z43"
9,asic_UK00,asic_UK00_3434,3434,80-130,F,<180,76-250,Overweight,0,<NA>,<NA>,10,<NA>,<NA>,<NA>,"A49,B37,B95,B96,C83,D62,E11,E87,I48,I50,J18,J91,J96,J98,K63,K65,K66,N39,R10,R63,R65,T81,U08,U09,U50,U69,U80,U81,U99,Z11,Z29,Z51,Z99"


### Rows Per Hospital

,hospital_id,rows
0,asic_UK00,10
1,asic_UK02,10
2,asic_UK03,10
3,asic_UK04,10
4,asic_UK06,10
5,asic_UK07,10
6,asic_UK08,10


### Source Map Sample

,hospital,canonical_name,raw_source_columns_used
0,asic_UK00,hospital_id,[derived from filename]
1,asic_UK00,stay_id_global,[derived from hospital_id and stay_id_local]
2,asic_UK00,stay_id_local,"[Pseudo-ID, PseudoID]"
3,asic_UK00,age_group,[clusterAlter]
4,asic_UK00,sex,[clusterGeschlecht]
5,asic_UK00,height_group,[clusterKoerpergroesse]
6,asic_UK00,weight_group,[clusterKoerpergewicht]
7,asic_UK00,bmi_group,[BMI]
8,asic_UK00,hosp_mortality,"[Entlassgrund_(verlegt_intern,_verlegt_extern,_verstorben)]"
9,asic_UK00,icu_mortality,[Sterblichkeit]


### Schema Summary

,hospital,rows,final_columns,empty_columns
0,asic_UK00,10,16,"[icu_mortality, hosp_los, icu_readmit, dialysis_free_days, vent_free_days]"
1,asic_UK02,10,16,[]
2,asic_UK03,10,16,"[icu_mortality, hosp_los, icu_readmit, dialysis_free_days, vent_free_days]"
3,asic_UK04,10,16,"[dialysis_free_days, vent_free_days]"
4,asic_UK06,10,16,[]
5,asic_UK07,10,16,[]
6,asic_UK08,10,16,"[hosp_los, dialysis_free_days, vent_free_days]"


## Final Dynamic Data

`final_dynamic` contains the harmonized dynamic table returned by the current pipeline.


In [4]:
display(Markdown("### Preview"))
display(final_dynamic.head(10))

display(Markdown("### Key Time-Series Columns"))
dynamic_preview_columns = [
    column
    for column in ["hospital_id", "stay_id", "time", "minutes_since_admit", "heart_rate", "map", "sbp", "dbp", "spo2", "resp_rate"]
    if column in final_dynamic.columns
]
display(final_dynamic[dynamic_preview_columns].head(20))

display(Markdown("### Rows Per Hospital"))
display(
    final_dynamic.groupby("hospital_id", dropna=False)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values("hospital_id")
    .reset_index(drop=True)
)

display(Markdown("### Schema Summary"))
display(dataset.dynamic.schema_summary)


### Preview

,hospital_id,stay_id_global,stay_id_local,time,minutes_since_admit,albumin,alt,amylase,ast,base_excess_art,bicarbonate_art,bilirubin_total,bnp,cardiac_index_bolus,cardiac_index_cont,cardiac_output_bolus,cardiac_output_cont,ck,ck_mb,clonidine_iv_cont,compliance,core_temp,creatinine,crp,cvp,d_dimer,dbp,delta_p,dexamethasone_iv_bolus,dexmedetomidine_iv_cont,dobutamine_iv_cont,ecmo,ecmo_o2,epinephrine_iv_cont,etco2,evlwi,extracorp_blood_flow,extracorp_o2_flow,fentanyl_iv_cont,feo2,fio2,fludrocortisone_po_bolus,fluid_balance_24h,furosemide_iv_cont,gedvi,heart_rate,hematocrit,hemoglobin,hydrocortisone_iv_bolus,ie_ratio,il6,inhaled_iloprost,inhaled_no,inr,insp_pressure,isoflurane_inh,ketanest_iv_cont,lactate_art,ldh,levosimendan_iv_cont,lipase,lymph_abs,lymph_pct,map,midazolam_iv_cont,milrinone_iv_cont,morphine_iv_cont,norepinephrine_iv_cont,ntprobnp,paco2,pao2,pap_dias,pap_mean,pap_sys,pct,pcwp,peep,pf_ratio,ph_art,platelets,position_therapy,prednisolone_iv_bolus,propofol_iv_cont,ptt,pvri,resp_rate,rocuronium_iv_bolus,sao2,sbp,scvo2,sevoflurane_inh,sofa,spo2,spont_resp_rate,stroke_index_bolus,stroke_index_cont,stroke_volume_bolus,stroke_volume_cont,sufentanil_iv_cont,svri,terlipressin_iv_bolus,troponin,urea,vasopressin_iv_cont,vt,vt_per_kg_ibw,wbc
0,asic_UK00,asic_UK00_372,372,0 days 00:00:00,0.0,NaN,NaN,NaN,NaN,1.3,26.9,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,39.7,8.00574,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.8,NaN,<NA>,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,<NA>,NaN,NaN,NaN,NaN,NaN,99.0,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
1,asic_UK00,asic_UK00_372,372,0 days 00:15:00,15.0,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
2,asic_UK00,asic_UK00_372,372,0 days 00:30:00,30.0,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
3,asic_UK00,asic_UK00_372,372,0 days 00:45:00,45.0,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
4,asic_UK00,asic_UK00_372,372,0 days 01:00:00,60.0,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
5,asic_UK00,asic_UK00_372,372,0 days 01:15:00,75.0,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,N

### Key Time-Series Columns

,hospital_id,time,minutes_since_admit,heart_rate,map,sbp,dbp,spo2,resp_rate
0,asic_UK00,0 days 00:00:00,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,asic_UK00,0 days 00:15:00,15.0,NaN,NaN,NaN,NaN,NaN,NaN
2,asic_UK00,0 days 00:30:00,30.0,NaN,NaN,NaN,NaN,NaN,NaN
3,asic_UK00,0 days 00:45:00,45.0,NaN,NaN,NaN,NaN,NaN,NaN
4,asic_UK00,0 days 01:00:00,60.0,NaN,NaN,NaN,NaN,NaN,NaN
5,asic_UK00,0 days 01:15:00,75.0,NaN,NaN,NaN,NaN,NaN,NaN
6,asic_UK00,0 days 01:30:00,90.0,NaN,NaN,NaN,NaN,NaN,NaN
7,asic_UK00,0 days 01:45:00,105.0,NaN,NaN,NaN,NaN,NaN,NaN
8,asic_UK00,0 days 02:00:00,120.0,NaN,NaN,NaN,NaN,NaN,NaN
9,asic_UK00,0 days 02:15:00,135.0,NaN,NaN,NaN,NaN,NaN,NaN


### Rows Per Hospital

,hospital_id,rows
0,asic_UK00,11656
1,asic_UK02,19297
2,asic_UK03,21942
3,asic_UK04,12684
4,asic_UK06,10808
5,asic_UK07,20455
6,asic_UK08,4805


### Schema Summary

,hospital,rows,final_columns,empty_columns
0,asic_UK00,11656,107,"[cardiac_index_bolus, cardiac_output_bolus, clonidine_iv_cont, dexamethasone_iv_bolus, dexmedetomidine_iv_cont, dobutamine_iv_cont, ecmo_o2, epinephrine_iv_cont, evlwi, extracorp_blood_flow, extra..."
1,asic_UK02,19297,107,"[bilirubin_total, bnp, cardiac_index_bolus, cardiac_output_bolus, delta_p, dexamethasone_iv_bolus, dexmedetomidine_iv_cont, extracorp_blood_flow, fentanyl_iv_cont, feo2, fludrocortisone_po_bolus, ..."
2,asic_UK03,21942,107,"[amylase, bnp, cardiac_index_bolus, cardiac_index_cont, cardiac_output_bolus, cardiac_output_cont, dbp, dexmedetomidine_iv_cont, dobutamine_iv_cont, ecmo_o2, epinephrine_iv_cont, evlwi, extracorp_..."
3,asic_UK04,12684,107,"[bnp, cardiac_output_cont, clonidine_iv_cont, d_dimer, dexamethasone_iv_bolus, dexmedetomidine_iv_cont, dobutamine_iv_cont, ecmo, ecmo_o2, epinephrine_iv_cont, extracorp_blood_flow, extracorp_o2_f..."
4,asic_UK06,10808,107,"[albumin, alt, amylase, ast, base_excess_art, bicarbonate_art, bilirubin_total, bnp, cardiac_index_bolus, cardiac_index_cont, cardiac_output_bolus, cardiac_output_cont, ck, ck_mb, clonidine_iv_con..."
5,asic_UK07,20455,107,"[bnp, cardiac_output_cont, ck_mb, compliance, extracorp_blood_flow, fentanyl_iv_cont, feo2, fludrocortisone_po_bolus, inhaled_iloprost, ketanest_iv_cont, levosimendan_iv_cont, pcwp, prednisolone_i..."
6,asic_UK08,4805,107,"[amylase, bnp, cardiac_index_bolus, cardiac_index_cont, cardiac_output_bolus, cardiac_output_cont, cvp, delta_p, ecmo, ecmo_o2, epinephrine_iv_cont, etco2, evlwi, extracorp_blood_flow, extracorp_o..."


## Pipeline QC Outputs

These are the additional pipeline artifacts generated alongside the final tables.


In [5]:
display(Markdown("### Static Categorical Value Summary"))
display(dataset.static.categorical_value_summary.head(30))

display(Markdown("### Dynamic Non-Numeric Issues"))
display(dataset.dynamic.non_numeric_issues)

display(Markdown("### Dynamic Distribution Summary"))
display(dataset.dynamic.distribution_summary.head(30))

display(Markdown("### Dynamic Distribution Issues"))
display(dataset.dynamic.distribution_issues)


### Static Categorical Value Summary

,hospital_id,column,value,count
0,asic_UK00,bmi_group,Normal Weight,1
1,asic_UK00,bmi_group,Obesity Class 3,1
2,asic_UK00,bmi_group,Overweight,8
3,asic_UK02,bmi_group,Normal Weight,2
4,asic_UK02,bmi_group,Obesity Class 1,1
5,asic_UK02,bmi_group,Obesity Class 3,2
6,asic_UK02,bmi_group,Overweight,4
7,asic_UK02,bmi_group,Underweight,1
8,asic_UK03,bmi_group,Normal Weight,2
9,asic_UK03,bmi_group,Overweight,7


### Dynamic Non-Numeric Issues

,hospital,raw_column,canonical_name,non_null_count,raw_non_numeric_count,resolved_by_custom_parser_count,unresolved_after_custom_parser_count,non_numeric_examples
0,asic_UK02,I:E,ie_ratio,957,957,957,0,"[1/1.5, 1/2.3, 1/1.4, 1.5/1, 1/1, 1/1.1, 1.1/1, 1/1.3, 1/1.7, 1/1.2]"
1,asic_UK04,Bilirubin_ges.,bilirubin_total,56,5,5,0,[<0.15]
2,asic_UK04,Lymphozyten_absolut,lymph_abs,14,4,4,0,[storniert]
3,asic_UK04,Lymphozyten_prozentual,lymph_pct,14,4,4,0,[storniert]
4,asic_UK04,CRP,crp,148,1,1,0,[<0.1]


### Dynamic Distribution Summary

,hospital,canonical_name,n,min,q1,median,q3,max,iqr,range_width
0,asic_UK00,albumin,57,225.699600,346.072720,376.166000,466.445840,526.632400,120.373120,300.932800
1,asic_UK02,albumin,177,0.000000,288.890000,352.090000,424.310000,552.210000,135.420000,552.210000
2,asic_UK03,albumin,123,184.830000,292.395000,328.755000,373.447500,559.035000,81.052500,374.205000
3,asic_UK07,albumin,154,0.780000,359.430000,390.585000,439.140000,555.080000,79.710000,554.300000
4,asic_UK00,alt,133,7.000000,21.000000,38.000000,61.000000,130.000000,40.000000,123.000000
5,asic_UK03,alt,212,6.599000,20.847000,35.694000,108.732000,6142.976000,87.885000,6136.377000
6,asic_UK04,alt,60,11.000000,16.000000,24.500000,55.750000,179.000000,39.750000,168.000000
7,asic_UK07,alt,228,6.000000,24.000000,39.000000,58.000000,328.000000,34.000000,322.000000
8,asic_UK08,alt,25,0.000000,18.000000,22.000000,36.000000,327.000000,18.000000,327.000000
9,asic_UK00,ast,138,9.000000,31.000000,46.000000,84.750000,356.000000,53.750000,347.000000


### Dynamic Distribution Issues

,canonical_name,hospital,n,flagged_metrics,min,median,iqr,max,range_width
0,albumin,asic_UK00,57,[max],225.699600,376.166000,120.373120,526.632400,300.932800
1,alt,asic_UK03,212,"[iqr, max, range_width]",6.599000,35.694000,87.885000,6142.976000,6136.377000
2,alt,asic_UK04,60,[min],11.000000,24.500000,39.750000,179.000000,168.000000
3,alt,asic_UK08,25,"[min, iqr]",0.000000,22.000000,18.000000,327.000000,327.000000
4,ast,asic_UK02,181,"[max, range_width]",12.000000,51.590000,75.590000,6180.560000,6168.560000
...,...,...,...,...,...,...,...,...,...
61,sufentanil_iv_cont,asic_UK08,2021,[iqr],0.000000,20.000000,47.470000,74.970000,74.970000
62,troponin,asic_UK02,25,"[min, iqr, max, range_width]",0.020000,0.290000,9.590000,15.080000,15.060000
63,urea,asic_UK08,52,[median],0.000000,22.000000,13.175000,44.100000,44.100000
64,vt_per_kg_ibw,asic_UK00,858,[min],5.822485,6.248122,0.248122,6.496243,0.673758


## Working Objects

The main notebook variables are:

- `dataset`: full pipeline result object
- `final_static`: final harmonized static table
- `final_dynamic`: final harmonized dynamic table
